In [1]:
import pandas as pd
import numpy as np
import folium

from sklearn.cluster import DBSCAN

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
df = pd.read_csv("../data/raw/ksp_synthetic_crime_dataset.csv")

print("Shape:", df.shape)

df.head()

Shape: (10000, 11)


,crime_id,district,police_station,crime_type,date,latitude,longitude,victim_age,offender_age,status,repeat_offender
0,CR000001,Kalaburagi,Kalaburagi PS 4,Vehicle Theft,2023-09-28,17.362084,76.910451,56,36,Closed,0
1,CR000002,Ballari,Ballari PS 5,Assault,2025-05-21,15.110356,76.895142,38,50,Charge Sheet Filed,0
2,CR000003,Tumakuru,Tumakuru PS 1,Cybercrime,2023-02-28,13.314309,77.095126,33,32,Under Investigation,0
3,CR000004,Belagavi,Belagavi PS 4,Robbery,2023-09-01,15.822481,74.503246,68,24,Open,0
4,CR000005,Mysuru,Mysuru PS 4,Theft,2023-11-12,12.194464,76.695371,43,19,Charge Sheet Filed,0


In [3]:
df = pd.read_csv("../data/raw/ksp_synthetic_crime_dataset.csv")

print("Shape:", df.shape)

df.head()

Shape: (10000, 11)


,crime_id,district,police_station,crime_type,date,latitude,longitude,victim_age,offender_age,status,repeat_offender
0,CR000001,Kalaburagi,Kalaburagi PS 4,Vehicle Theft,2023-09-28,17.362084,76.910451,56,36,Closed,0
1,CR000002,Ballari,Ballari PS 5,Assault,2025-05-21,15.110356,76.895142,38,50,Charge Sheet Filed,0
2,CR000003,Tumakuru,Tumakuru PS 1,Cybercrime,2023-02-28,13.314309,77.095126,33,32,Under Investigation,0
3,CR000004,Belagavi,Belagavi PS 4,Robbery,2023-09-01,15.822481,74.503246,68,24,Open,0
4,CR000005,Mysuru,Mysuru PS 4,Theft,2023-11-12,12.194464,76.695371,43,19,Charge Sheet Filed,0


In [4]:
df.columns

Index(['crime_id', 'district', 'police_station', 'crime_type', 'date',
       'latitude', 'longitude', 'victim_age', 'offender_age', 'status',
       'repeat_offender'],
      dtype='str')

In [5]:
coords = df[['latitude','longitude']]
coords.head()

,latitude,longitude
0,17.362084,76.910451
1,15.110356,76.895142
2,13.314309,77.095126
3,15.822481,74.503246
4,12.194464,76.695371


In [6]:
dbscan = DBSCAN(
    eps=0.02,
    min_samples=15
)

df['cluster'] = dbscan.fit_predict(coords)

df.head()

,crime_id,district,police_station,crime_type,date,latitude,longitude,victim_age,offender_age,status,repeat_offender,cluster
0,CR000001,Kalaburagi,Kalaburagi PS 4,Vehicle Theft,2023-09-28,17.362084,76.910451,56,36,Closed,0,0
1,CR000002,Ballari,Ballari PS 5,Assault,2025-05-21,15.110356,76.895142,38,50,Charge Sheet Filed,0,1
2,CR000003,Tumakuru,Tumakuru PS 1,Cybercrime,2023-02-28,13.314309,77.095126,33,32,Under Investigation,0,2
3,CR000004,Belagavi,Belagavi PS 4,Robbery,2023-09-01,15.822481,74.503246,68,24,Open,0,3
4,CR000005,Mysuru,Mysuru PS 4,Theft,2023-11-12,12.194464,76.695371,43,19,Charge Sheet Filed,0,-1


In [7]:
hotspot_centroids = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    .agg(
        centroid_latitude=("latitude", "mean"),
        centroid_longitude=("longitude", "mean"),
        crime_count=("cluster", "count")
    )
    .reset_index()
)

hotspot_centroids

,cluster,centroid_latitude,centroid_longitude,crime_count
0,0,17.330638,76.832556,879
1,1,15.140800,76.923906,900
2,2,13.338785,77.101543,937
3,3,15.852962,74.496785,865
4,4,12.913223,74.853127,947
5,5,12.297595,76.638324,923
6,6,16.830567,75.709875,887
7,7,13.928169,75.566613,952
8,8,15.459897,75.007333,944
9,9,12.975495,77.597000,917


In [8]:
hotspot_centroids.to_csv(
    "../data/processed/hotspot_centroids.csv",
    index=False
)

print("hotspot_centroids.csv saved successfully")

hotspot_centroids.csv saved successfully


In [9]:
df['cluster'].value_counts()

cluster
 7    952
 4    947
 8    944
 2    937
 5    923
 9    917
 1    900
 6    887
 0    879
 3    865
-1    849
Name: count, dtype: int64

In [10]:
hotspot_centroids.head()

,cluster,centroid_latitude,centroid_longitude,crime_count
0,0,17.330638,76.832556,879
1,1,15.140800,76.923906,900
2,2,13.338785,77.101543,937
3,3,15.852962,74.496785,865
4,4,12.913223,74.853127,947


In [11]:
num_hotspots = len(set(df['cluster'])) - (1 if -1 in df['cluster'].values else 0)

print("Total Hotspots:", num_hotspots)

Total Hotspots: 10


In [12]:
m = folium.Map(
    location=[
        df.latitude.mean(),
        df.longitude.mean()
    ],
    zoom_start=7
)

In [13]:
for _, row in df.iterrows():

    if row['cluster'] != -1:

        folium.CircleMarker(
            location=[
                row['latitude'],
                row['longitude']
            ],
            radius=3,
            popup=f"Cluster {row['cluster']}",
            fill=True
        ).add_to(m)

In [14]:
m.save("../gis/outputs/maps/crime_hotspots.html")

print("Hotspot map saved successfully!")

Hotspot map saved successfully!


In [16]:
hotspots = (
    df[df['cluster'] != -1]
    .groupby('cluster')
    .size()
    .reset_index(name='crime_count')
)

hotspots

,cluster,crime_count
0,0,879
1,1,900
2,2,937
3,3,865
4,4,947
5,5,923
6,6,887
7,7,952
8,8,944
9,9,917


In [17]:
hotspots.to_csv(
    "../data/processed/crime_hotspots.csv",
    index=False
)